In [15]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch
import sys
import os
sys.path.append(os.path.abspath(".."))
from core_ntsa.predicting import predict_zeroth_order, predict_first_order, simplex_projection
from core_ntsa.generators import simulate_lorenz
from core_ntsa.noise_tools import add_white_noise, add_colored_noise
from core_ntsa.surrogates import generate_aaft_surrogates, generate_iaaft_surrogates
from core_ntsa.metrics import calculate_correlation_dimension, calculate_kantz_divergence, calculate_rosenstein_divergence, calculate_wolf_lle
from core_ntsa.statistic_test import one_sided_rank_test, run_surrogate_pipeline
lorenz_data = simulate_lorenz(t_span=50.0, dt=0.01)
clean_x = lorenz_data[0]

x_white_noise = add_white_noise(clean_x, snr_db=20.0)
x_clored_noise = add_colored_noise(clean_x, color='pink', snr_db=10.0)

In [16]:
if __name__ == '__main__':
    # ==========================================
    # KHUNG KIEM THU (TESTING BLOCK)
    # ==========================================
    
    # Do ham predict_zeroth_order cua ban tra ve 3 bien (normalized_error, y_true, y_pred),
    # ta can mot wrapper nho de chi trich xuat dai luong normalized_error (gia tri vo huong) cho pipeline.
    def zeroth_order_error_only(sig, m, tau, horizon):
        error, _, _ = predict_zeroth_order(sig, m, tau, horizon)
        return error


    # Tao du lieu gia lap
    
    # Thiet lap tham so
    m_opt = 3
    tau_opt = 16
    horizon_opt = 1
    M_surr = 39

    # Chay pipeline
    try:
        final_result = run_surrogate_pipeline(
            signal=x_white_noise,
            surrogate_generator=generate_iaaft_surrogates,
            evaluator=zeroth_order_error_only,
            M=M_surr,
            test_direction='less', # RMSE cua du lieu that can nho hon
            surrogate_kwargs={'max_iter': 50, 'random_seed': 42},
            evaluator_kwargs={'m': m_opt, 'tau': tau_opt, 'horizon': horizon_opt}
        )
        
        print("Kiem dinh thanh cong!")
        for key, val in final_result.items():
            print(f"{key}: {val}")
            
    except Exception as e:
        print(f"Loi thuc thi: {e}")

Kiem dinh thanh cong!
t_orig: 0.20005050855943313
t_surr_mean: 0.259866170732114
t_surr_std: 0.02920968228266959
p_value: 0.025
surrogates_beat_orig: 0
M: 39
test_direction: less
